# 大規模言語モデルの中身を覗いてみる
## Qwen3-8B で学ぶ "感情ベクトル" 入門

メディア研究を専攻する学部生のための導入講義ノートブック  
担当: 楊鯤昊 (芝浦工業大学)

---

このノートブックでは、皆さんが普段使っている ChatGPT のような **大規模言語モデル (LLM)** の "中身" を実際に開けて、観察してみます。今日のゴールは、次の問いに自分の手で答えを出すことです:

> **「LLM は『怒り』や『悲しみ』のような抽象的な感情を、内部でどのように表現しているのだろうか?」**

最終的に、皆さんは **感情ベクトル** という 4096 次元の数列を取り出し、それが本当に意味を持っていることを自分の目で確かめます。

### このノートブックの読み方

このノートブックは **「読みもの」としても、「実験ノート」としても** 使えるように作られています。

- **コードに興味がない方へ**: コードセルは折りたたまれています。各セルの ▶ ボタンを押して実行するだけで、結果や図が表示されます。本文と図だけ追っていけば、講義の流れは完全に追えます。
- **コードに興味がある方へ**: 折りたたまれているタイトルバーをクリックすると、中のコードが展開されます。値を変えて再実行し、自分の手で実験を広げてください。

### 前提知識

- Python の経験は **不要** です (が、あれば理解が深まります)
- 高校レベルの数学 (ベクトル、平均) を知っていれば十分です
- Google アカウントがあれば、Google Colab で直接実行できます

### 参考文献

- Anthropic 研究記事 (2026年4月): [Emotion Concepts and Their Function in a Large Language Model](https://www.anthropic.com/research/emotion-concepts-function)
- Qwen3-8B モデルカード: [huggingface.co/Qwen/Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B)
- LLMs入門論文: Hussain, Z., Binz, M., Mata, R. et al. (2024). [A tutorial on open-source large language models for behavioral science.](https://doi.org/10.3758/s13428-024-02455-8) Behav. Res. 56, 8214–8237
- Anthropic 研究批判 (2026年4月): [Latent Space or Weight Space: Which One Is Primary](https://substack.com/@deepmanifold/p-192478658)
- 本講義のリポジトリ: [github.com/CSS-Laboratory/LLMintro](https://github.com/CSS-Laboratory/LLMintro)

---

### 講義の流れ (5 つのセクション)

1. **準備運動: LLM に届く前の私たちの言葉** ― 「思考モード」のオン・オフを比較
2. **ニューラルネットワーク入門** ― なぜ層を重ねると賢くなるのか
3. **Forward Hook: モデルの中を覗く窓** ― 隠れ状態を取り出す技術
4. **感情ベクトルを抽出する** ― 280 個のストーリーから "怒り方向" を計算
5. **感情ベクトルは何を表現しているのか?** ― 幾何学と検証

それでは、始めましょう。


## ⚙️ ステップ 0: 環境セットアップ

最初に、必要なライブラリをインストールします。Colab の **無料 T4 GPU** ランタイムを使用してください (上部メニュー: ランタイム → ランタイムのタイプを変更 → T4 GPU)。

下のセルを実行すると 2-3 分ほどかかります。エラーが出ても続行で大丈夫です — もし 2 回目以降で `ImportError` が出たら、メニューから「ランタイム → セッションを再起動」してから再実行してください。


In [ ]:
#@title 環境セットアップ (実行してください) {display-mode: "form"}
import subprocess, sys, os, gc
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# Pinned versions known to work with Qwen3-8B on Colab T4 (Nov 2025)
PACKAGES = [
    "transformers>=4.51.0,<5.0.0",
    "huggingface_hub<1.0.0",
    "tokenizers>=0.21.0,<0.22.0",
    "accelerate",
    "bitsandbytes",
    "datasets",
    "scikit-learn",
    "matplotlib",
    "japanize-matplotlib",
    "pandas",
    "tqdm",
    "sentencepiece",
]

if IN_COLAB:
    print("Colab を検出しました。必要なライブラリをインストールしています...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-U", *PACKAGES]
    )
    print("✅ インストール完了")

# Imports
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import japanize_matplotlib  # 日本語フォントを matplotlib で使う
from tqdm.auto import tqdm
import urllib.request, json

# 再現性のための乱数シード
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# パスとデバイス
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CACHE_DIR = Path("/content/llmintro_cache") if IN_COLAB else Path.cwd() / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# GitHub のキャッシュデータをダウンロードする URL
GITHUB_DATA_BASE = "https://raw.githubusercontent.com/CSS-Laboratory/LLMintro/main/data"

def download_from_github(filename):
    """GitHub の data/ フォルダからファイルをダウンロード"""
    url = f"{GITHUB_DATA_BASE}/{filename}"
    out = CACHE_DIR / filename
    if not out.exists():
        print(f"  ダウンロード中: {filename}")
        urllib.request.urlretrieve(url, out)
    return out

print(f"\nデバイス: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")


### 実行モードの選択

以下のセルで実行モードを選べます。

- `LIVE_MODE = True` (デフォルト): モデルを実際に動かします。各セクションを自分の目で確かめられます。
- `LIVE_MODE = False`: 事前計算済みの結果を読み込むだけのデモモード。コードを動かさず流れだけ追いたい方向け。

時間がない場合や、Colab で問題が起きた場合は `False` に変えて再実行してください。


In [ ]:
#@title 実行モードを設定 {display-mode: "form"}
LIVE_MODE = True  #@param {type:"boolean"}

if LIVE_MODE:
    print("✅ ライブモード: モデルを実際に動かします (推奨)")
else:
    print("📦 デモモード: 事前計算済みのデータを読み込みます")


---

## 📍 セクション 1: LLM に届く前の私たちの言葉

### 比喩: レストランの注文票

私たちが ChatGPT に「今日は天気がいいね」と打ち込むとき、モデルはその文字列をそのまま読んでいるわけではありません。

レストランで料理を頼む場面を想像してください。

> お客さん「ハンバーグセットを 1 つ。サラダはドレッシング別で」  
> ホールスタッフ → キッチン伝票: `[T03] HBG×1 / SAL-DRSG sep`

お客さんの自然な言葉と、キッチンに渡る "業務用フォーマット" は別物です。LLM の世界でもまったく同じ仕組みが使われています。これを **チャットテンプレート (chat template)** と呼びます。

このセクションでは、その "業務用フォーマット" が実際にどう見えるかを覗いてみます。


In [ ]:
#@title トークナイザを読み込み (約 10 秒) {display-mode: "form"}
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen3-8B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR / "models")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print("✅ トークナイザを読み込みました")
print(f"  語彙サイズ: {len(tokenizer):,} トークン")


### 「思考モード」オフ vs オン

Qwen3 には 2 つのモードがあります。

- **思考オフ** (`enable_thinking=False`): 質問にすぐ答える。素早いが、複雑な推論は苦手。
- **思考オン** (`enable_thinking=True`): 答える前に「下書き」を書く。出力に `<think>...</think>` ブロックが含まれる。

私たちの新聞記者プロンプトで、両者がどう変わるかを見てみましょう。

まずは **モデルが実際に受け取る文字列** だけを見比べます。


In [ ]:
#@title チャットテンプレートを描画 (思考オフ vs 思考オン) {display-mode: "form"}
demo_messages = [
    {"role": "system", "content": "あなたは経験豊富な新聞記者です。"},
    {"role": "user", "content": "猛暑の長期化について、見出しを1つ書いてください。"},
]

rendered_off = tokenizer.apply_chat_template(
    demo_messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
)
rendered_on = tokenizer.apply_chat_template(
    demo_messages, tokenize=False, add_generation_prompt=True, enable_thinking=True,
)

print("=" * 70)
print("【思考モード OFF: モデルが受け取る生の文字列】")
print("=" * 70)
print(rendered_off)
print()
print("=" * 70)
print("【思考モード ON: モデルが受け取る生の文字列】")
print("=" * 70)
print(rendered_on)


### 観察

`<|im_start|>` や `<|im_end|>` のような **特殊トークン** が見えますね。これは「ここから system の発言です」「ここでユーザの発言が終わりです」という業務用ラベルです。

思考モードがオンのときは、`<think>` という空のブロックが追加されているのがわかります。これがモデルへの「ここで下書きをしていいよ」という合図です。

> 💡 **メディア研究的視点**: 私たちが普段使っている自然言語は、こうした不可視の構造を通してモデルに渡されています。これは Erving Goffman の「フレーム分析」が言うところの、**コミュニケーションの不可視な舞台装置** のデジタル版とも言えます。


### 実際に生成してみる

では、両モードで実際にテキストを生成してみましょう。先にモデル本体を読み込みます (4-bit 量子化を使うので VRAM は約 6GB)。所要時間は約 2 分です。


In [ ]:
#@title Qwen3-8B 本体を読み込み (4-bit 量子化、約 2 分) {display-mode: "form"}
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

if LIVE_MODE:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        cache_dir=CACHE_DIR / "models",
        quantization_config=quant_config,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    model.eval()

    NUM_LAYERS = len(model.model.layers)
    HIDDEN_SIZE = model.config.hidden_size

    if torch.cuda.is_available():
        used_gb = torch.cuda.memory_allocated() / 1e9
        print(f"\n✅ モデル読み込み完了")
        print(f"  使用 VRAM: {used_gb:.2f} GB")
        print(f"  Transformer 層の数: {NUM_LAYERS}")
        print(f"  隠れ状態の次元: {HIDDEN_SIZE}")
else:
    model = None
    NUM_LAYERS = 36
    HIDDEN_SIZE = 4096
    print("📦 デモモードのためモデルは読み込みません。")


In [ ]:
#@title 思考オフ vs 思考オンで生成 (約 30 秒) {display-mode: "form"}
def generate_text(messages, enable_thinking, max_new_tokens=128):
    """シンプルなテキスト生成ヘルパー"""
    rendered = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking,
    )
    inputs = tokenizer(rendered, return_tensors="pt", add_special_tokens=False).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            do_sample=True,
            temperature=0.6 if enable_thinking else 0.7,
            top_p=0.95 if enable_thinking else 0.8,
            top_k=20,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=False)

if LIVE_MODE:
    print("【思考モード OFF の出力】")
    print("-" * 50)
    print(generate_text(demo_messages, enable_thinking=False, max_new_tokens=80))

    print("\n【思考モード ON の出力】")
    print("-" * 50)
    print(generate_text(demo_messages, enable_thinking=True, max_new_tokens=300))
else:
    print("📦 デモモード: 生成はスキップしました")


### まとめ

思考モードでは、モデルが **下書きを書いてから清書する** イメージです。下書きは `<think>...</think>` ブロックの中に見えています。これは出力の質を変えますが、内部の表現自体は同じトランスフォーマー層から取り出します。

このノートブックの以降では、シンプルさのため **思考モードはオフ** で進めます。皆さんの目的は出力を見ることではなく、**モデル内部の隠れ状態を観察する** ことだからです。


---

## 📍 セクション 2: ニューラルネットワーク入門 ― なぜ層を重ねると賢くなるのか?

### 比喩: 折り紙

机の上にリンゴ 🍎 と蜜柑 🍊 がバラバラに置かれているとします。1 枚の真っ直ぐな下敷き (= **線形分類器**) では、リンゴと蜜柑を 1 本の直線でしか分けられません。並びがちょっと複雑なだけで、もう失敗します。

でも、紙を **何度も折りたためば**、複雑にぐにゃぐにゃ曲がった境界線で分けることができます。

ニューラルネットワークの「層を重ねる」とは、まさにこの **折りたたみを何度も繰り返す** ことです。Qwen3-8B には 36 段の折りたたみが入っています。

実際に、層を 1 → 2 → 3 → 5 と増やしながら、何が起きるかを見てみましょう。


In [ ]:
#@title 図 1: 線形分類器の限界 {display-mode: "form"}
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression

# 三日月型の 2 クラスデータ (線形分離不可能)
X, y = make_moons(n_samples=400, noise=0.20, random_state=SEED)

# 線形分類器 (logistic regression) を学習
linear_clf = LogisticRegression().fit(X, y)
linear_acc = linear_clf.score(X, y)

# 描画用グリッド
xx, yy = np.meshgrid(np.linspace(-1.5, 2.5, 200), np.linspace(-1.0, 1.5, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
zz = linear_clf.predict_proba(grid)[:, 1].reshape(xx.shape)

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.contourf(xx, yy, zz, levels=20, cmap="RdBu_r", alpha=0.6)
ax.contour(xx, yy, zz, levels=[0.5], colors="k", linewidths=2)
ax.scatter(X[y == 0, 0], X[y == 0, 1], c="#cc4444", s=20, edgecolor="white", label="クラス 0")
ax.scatter(X[y == 1, 0], X[y == 1, 1], c="#4444cc", s=20, edgecolor="white", label="クラス 1")
ax.set_title(f"線形分類器の決定境界 (精度: {linear_acc:.1%})", fontsize=13)
ax.legend(loc="upper right")
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()
plt.show()

print(f"\n線形分類器では、この曲がったデータは {linear_acc:.0%} しか正しく分けられません。")
print("直線では限界があります。次に、層を重ねた MLP を試してみましょう。")


### 層を増やしていく

下の図では、**MLP (Multi-Layer Perceptron)** という最もシンプルなニューラルネットワークの隠れ層を 1 → 2 → 3 → 5 と増やしていきます。各セルは、500 ステップの学習後の決定境界を表示します。

注目してほしいのは、**層が深くなるほど境界線が滑らかに曲がる** ことです。


In [ ]:
#@title 図 2: MLP の層を増やすとどうなるか {display-mode: "form"}
import torch.nn as nn

def make_mlp(num_hidden_layers, hidden_dim=16):
    """指定された層数の MLP を構築"""
    layers = [nn.Linear(2, hidden_dim), nn.ReLU()]
    for _ in range(num_hidden_layers - 1):
        layers.append(nn.Linear(hidden_dim, hidden_dim))
        layers.append(nn.ReLU())
    layers.append(nn.Linear(hidden_dim, 2))
    return nn.Sequential(*layers)

def train_mlp(net, X, y, steps=500, lr=0.05):
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.long)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(steps):
        optimizer.zero_grad()
        loss = loss_fn(net(X_t), y_t)
        loss.backward()
        optimizer.step()
    with torch.no_grad():
        acc = (net(X_t).argmax(dim=1) == y_t).float().mean().item()
    return net, acc

depths = [1, 2, 3, 5]
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
axes = axes.flatten()

for ax, depth in zip(axes, depths):
    torch.manual_seed(SEED)  # 各深さで同じ初期化
    net, acc = train_mlp(make_mlp(depth), X, y, steps=500)
    grid_t = torch.tensor(grid, dtype=torch.float32)
    with torch.no_grad():
        probs = torch.softmax(net(grid_t), dim=1)[:, 1].numpy().reshape(xx.shape)

    ax.contourf(xx, yy, probs, levels=20, cmap="RdBu_r", alpha=0.6)
    ax.contour(xx, yy, probs, levels=[0.5], colors="k", linewidths=2)
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c="#cc4444", s=18, edgecolor="white")
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c="#4444cc", s=18, edgecolor="white")
    ax.set_title(f"隠れ層 {depth} 層    精度: {acc:.1%}", fontsize=12)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle("層を増やすと、境界線が「折りたためる」", fontsize=14, y=1.00)
plt.tight_layout()
plt.show()


### 接続: Qwen3-8B はこれを 36 回やる

Qwen3-8B には **36 個** のトランスフォーマー層が積まれています。各層は、上の MLP の例よりずっと複雑な「折りたたみ」を、文章全体の文脈を吸収しながら行います。

実際の構造を見てみましょう。


In [ ]:
#@title 図 3: Qwen3-8B のレイヤー構造を覗く {display-mode: "form"}
if LIVE_MODE:
    print(f"Qwen3-8B のレイヤー総数: {NUM_LAYERS}")
    print(f"各レイヤーの入出力次元 (= 隠れ状態の次元): {HIDDEN_SIZE}\n")
    print("最初のレイヤーの構造:")
    print("=" * 60)
    print(model.model.layers[0])
else:
    print("📦 デモモード: NUM_LAYERS = 36, HIDDEN_SIZE = 4096")


この出力を読み解くと:

- `Qwen3DecoderLayer` ← 1 つの「折りたたみ」を担う部品
- `self_attn` ← **自己注意機構**: 文中の各単語が他のどの単語に注目すべきかを決める
- `mlp` ← **フィードフォワードネットワーク**: 上で見た MLP と同じ役割

要するに、**1 つの層 = 「文中の関係性を見渡す (attention)」+「非線形に折りたたむ (mlp)」**。これを 36 回繰り返すのが Qwen3-8B です。

### "Large" って何が大きいのか?

LLM の "L" は **Large = 大きい** を意味します。何が大きいか?それは **パラメータの数** です。1 つのパラメータを 1 円玉に例えるなら、80 億円 (= 80 億パラメータ) が銀行の中で互いに連携しながら「次の単語」を計算しているイメージです。


In [ ]:
#@title 図 4: 主要な LLM のパラメータ数比較 {display-mode: "form"}
models_compare = {
    "BERT-base\n(2018)": 0.110,
    "GPT-2 small\n(2019)": 0.124,
    "GPT-2 XL\n(2019)": 1.5,
    "Qwen3-8B\n(2025, 今日のモデル)": 8.0,
    "Llama-3 70B\n(2024)": 70.0,
    "GPT-4 (推定)\n(2023)": 1700.0,
}

names = list(models_compare.keys())
values = list(models_compare.values())
colors = ["#888888"] * len(names)
colors[3] = "#d43f3a"  # Qwen3-8B を強調

fig, ax = plt.subplots(figsize=(10, 4.5))
bars = ax.barh(names, values, color=colors, edgecolor="white")
ax.set_xscale("log")
ax.set_xlabel("パラメータ数 (10 億単位、対数軸)", fontsize=11)
ax.set_title("主要な LLM のパラメータ数比較", fontsize=13)

for bar, val in zip(bars, values):
    label = f"{val:.1f}B" if val < 1000 else f"{val/1000:.1f}T"
    ax.text(val * 1.1, bar.get_y() + bar.get_height()/2, label,
            va="center", fontsize=10)

ax.set_xlim(0.05, 5000)
plt.tight_layout()
plt.show()

print("対数軸であることに注意してください。GPT-4 は Qwen3-8B の約 200 倍のパラメータがあります。")
print("一方、本講義で扱う Qwen3-8B でも、内部表現の本質的な現象を観察するには十分です。")


---

## 📍 セクション 3: Forward Hook ― モデルの中を覗く窓

### 比喩: パイプの途中に温度計を取り付ける

Qwen3-8B は **36 段のパイプライン** のようなもので、文章は 1 段ずつ通り抜けながら、各段で「折りたたまれて」いきます。

**Forward Hook (フォワードフック)** は、このパイプの途中 (例: 24 段目) に温度計を差し込んで、そこを通る数字を盗み聞きする仕掛けです。モデル本来の流れを邪魔せず、**観察だけ** をします。

PyTorch の機能で、わずか 1 行のコードで取り付けられます。


In [ ]:
#@title 図 5: Hook の仕組みを最小限の例で確認 {display-mode: "form"}
import torch.nn as nn

# おもちゃの線形層 (3 次元 → 2 次元)
toy_layer = nn.Linear(in_features=3, out_features=2)
captured_data = None

def my_hook(module, inputs, output):
    """この関数が、forward 計算のたびに自動で呼ばれる"""
    global captured_data
    print(f"  🌡️ Hook 発動: 層を 1 回通った")
    print(f"     入力 shape: {inputs[0].shape}")
    print(f"     出力 shape: {output.shape}")
    captured_data = output.detach().clone()

# Hook を取り付ける
handle = toy_layer.register_forward_hook(my_hook)

# 順方向計算を実行
fake_input = torch.tensor([1.0, 2.0, 3.0])
print("forward を実行...")
result = toy_layer(fake_input)

# Hook を外す (お行儀よく後片付け)
handle.remove()

print(f"\n捕まえた値: {captured_data.numpy()}")
print(f"これは forward の出力 = {result.detach().numpy()} と完全に一致しています。")


### Qwen3-8B に取り付ける

同じ仕組みを、本物の Qwen3-8B に取り付けます。36 層あるうち、**66% の位置 (24 層目)** にフックを置きます。

なぜ 24 層目?
- 浅い層 (0〜8 層) は、文字や単語の表層的な情報を扱う
- 深い層 (28〜35 層) は、次の単語を予測することに特化しすぎる
- **中間〜後半 (24 層目あたり)** が、もっとも **抽象的・概念的** な情報を持っている

これは多くの LLM 解釈研究で経験的に確認されています。


In [ ]:
#@title Qwen3-8B の 24 層目に Hook を取り付け {display-mode: "form"}
if LIVE_MODE:
    LAYER_TO_PROBE = int(round(NUM_LAYERS * 2 / 3))  # 36 * 2/3 = 24
    print(f"対象レイヤー: {LAYER_TO_PROBE} / {NUM_LAYERS} 層中 (66% の位置)\n")

    # 既存の hook をクリア (再実行時の事故防止)
    model.model.layers[LAYER_TO_PROBE]._forward_hooks.clear()

    captured_hidden = None

    def capture_last_token(module, inputs, output):
        """最後のトークンに対応する隠れ状態だけを保存"""
        global captured_hidden
        # output はタプルのこともあるので unpack
        hidden = output[0] if isinstance(output, tuple) else output
        captured_hidden = hidden[:, -1, :].detach().clone()

    handle = model.model.layers[LAYER_TO_PROBE].register_forward_hook(capture_last_token)

    # テスト文を流す
    test_text = "I am so unbelievably angry right now."
    inputs = tokenizer(test_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        _ = model(**inputs)

    handle.remove()

    print(f"入力した文: 「{test_text}」")
    print(f"取り出した隠れ状態の shape: {tuple(captured_hidden.shape)}")
    print(f"  → (バッチサイズ 1, 隠れ次元 {HIDDEN_SIZE})")
    print(f"値の絶対値の平均: {captured_hidden.abs().mean().item():.3f}")
else:
    LAYER_TO_PROBE = 24
    print("📦 デモモード: LAYER_TO_PROBE = 24 (24 / 36 層目)")


### 観察

**4096 次元のベクトル** が取れました。これが「この文を読んだ瞬間の、モデル 24 層目での内部状態」です。

ただし、この 4096 個の数字を単体で眺めても、人間にとっての意味はまだ見えません。

> 1 人の脳波を 1 回測っただけでは「この人は今何を感じているか」は分かりません。でも **「怒っているとき」と「穏やかなとき」を何度も比較する** と、両者を区別する **方向** が見えてきます。

これが次のセクションでやることです。


---

## 📍 セクション 4: 感情ベクトルを抽出する

### 比喩: 群衆写真の差分

50 人の「怒っている人」の集合写真を **重ね合わせて平均化** すると、個別の顔は消えて、共通する特徴 (眉間のしわ、強い口元) だけが残ります。

同じことを **モデル内部の活性化 (隠れ状態)** に対して行います。

1. 「怒りを表現する文」を **大量に** 用意する (このノートブックでは 1 感情あたり約 20 個)
2. 各文を Qwen3-8B に流して、24 層目の隠れ状態を取り出す
3. それらを平均する → 「怒り方向の代表ベクトル」
4. 同じことを「穏やかな文」「悲しい文」… と全部の感情でやる
5. 全感情の平均 (中心) を引き算する → **きれいな "感情方向" のベクトル**

これを **Difference of Means (平均差分法)** と呼びます。Anthropic が 2026 年 4 月の論文 ([リンク](https://www.anthropic.com/research/emotion-concepts-function)) で使った手法と本質的に同じです。


### 数式 (興味のある人向け)

$N$ 個の「感情 $e$ を表す文」の隠れ状態を $h(\text{text}_i^e)$ とすると、

$$
\mathbf{v}_e = \frac{1}{N} \sum_{i=1}^{N} h(\text{text}_i^e) - \boldsymbol{\mu}_{\text{global}}
$$

ここで $\boldsymbol{\mu}_{\text{global}}$ は全感情の隠れ状態の平均 (= グローバル中心)。グローバル中心を引くことで、**「どの感情を読むときも共通する成分」(例: 文法、書き手の癖)** が消え、**感情そのものの違いだけ** が残ります。

これは線形代数で言う「中心化 (centering)」です。多変量解析の主成分分析 (PCA) の前処理と同じ発想です。


### ストーリーをダウンロード

以下のセルで、事前に Qwen3-8B が生成した **280 個のストーリー** (14 感情 × 20 個) を GitHub からダウンロードします。これらは皆さんの講師が事前に生成した結果です。

- ストーリーは 14 種類の感情 (`happy`, `sad`, `angry`, `afraid`, `calm`, `loving`, `hostile`, `desperate`, `surprised`, `proud`, `gloomy`, `reflective`, `blissful`, `enthusiastic`) で構成されています。
- 重要な点: ストーリー本文の中で **感情語そのものは使っていません**。「怒り」を表現するストーリーであっても、`angry` という単語は出てきません。これは「単語の暗記」ではなく **本物の感情概念** をモデルが捉えているかを検証するためです。


In [ ]:
#@title ストーリーキャッシュをダウンロード {display-mode: "form"}
import json

stories_path = download_from_github("stories_smoke.jsonl")

stories = []
with stories_path.open() as f:
    for line in f:
        stories.append(json.loads(line))

stories_df = pd.DataFrame(stories)
print(f"✅ {len(stories_df)} 個のストーリーを読み込みました")
print(f"  感情の種類: {stories_df['emotion'].nunique()} 種類")
print(f"  各感情のストーリー数: {stories_df.groupby('emotion').size().describe()['mean']:.0f} 個 (平均)")
print()

# 感情ごとのカウント
print(stories_df.groupby("emotion").size().rename("count").to_frame().T)


### ストーリーを実際に読んでみる

3 つの感情 (`angry`, `happy`, `gloomy`) について、ストーリー本文を 1 つずつ表示します。**感情語が直接使われていない** ことを確かめてください。


In [ ]:
#@title サンプルストーリーを表示 {display-mode: "form"}
from IPython.display import HTML, display

samples = []
for emo in ["angry", "happy", "gloomy"]:
    row = stories_df[stories_df["emotion"] == emo].iloc[0]
    samples.append({"感情ラベル": emo, "ストーリー本文 (英語)": row["text"][:500] + ("..." if len(row["text"]) > 500 else "")})

samples_df = pd.DataFrame(samples)

# HTML で見やすく表示
html = "<table style='border-collapse: collapse; width: 100%; font-family: sans-serif;'>"
html += "<tr style='background: #2d3748; color: white;'>"
html += "<th style='padding: 10px; text-align: left; width: 15%;'>感情ラベル</th>"
html += "<th style='padding: 10px; text-align: left;'>ストーリー本文 (英語)</th></tr>"
for _, r in samples_df.iterrows():
    html += f"<tr style='border-bottom: 1px solid #ccc;'>"
    html += f"<td style='padding: 10px; vertical-align: top; font-weight: bold; color: #d43f3a;'>{r['感情ラベル']}</td>"
    html += f"<td style='padding: 10px; vertical-align: top; line-height: 1.6;'>{r['ストーリー本文 (英語)']}</td></tr>"
html += "</table>"
display(HTML(html))


### 280 ストーリー全部の隠れ状態を抽出する

ここがこのノートブックで **一番計算が重い** ステップです。T4 GPU で 5〜10 分ほどかかります。

途中で進捗バーが表示されるので、コーヒーでも飲みながら待ちましょう ☕

問題が起きた場合は、`LIVE_MODE = False` に設定して再実行すると、事前計算済みの結果を読み込みます。


In [ ]:
#@title ストーリー全部の隠れ状態を抽出 (5〜10 分) {display-mode: "form"}
def extract_story_activations(texts, layer_idx, batch_size=4, pool_skip_first=4):
    """
    各テキストを model に流し、layer_idx の隠れ状態を pool_skip_first 以降のトークンで平均する。
    シンプルさを優先した実装。
    """
    model.model.layers[layer_idx]._forward_hooks.clear()

    captured = {}
    def hook(module, inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        captured["seq"] = hidden.detach().clone()

    handle = model.model.layers[layer_idx].register_forward_hook(hook)
    pooled_list = []

    tokenizer.padding_side = "right"  # 末尾まで読みたいので右パディング

    for i in tqdm(range(0, len(texts), batch_size), desc="活性化を抽出中"):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(model.device)
        with torch.no_grad():
            _ = model(**inputs)
        hidden = captured["seq"].float().cpu()
        attention_mask = inputs["attention_mask"].cpu()

        for row in range(hidden.shape[0]):
            seq_len = int(attention_mask[row].sum().item())
            start = min(pool_skip_first, seq_len - 1)
            pooled = hidden[row, start:seq_len].mean(dim=0).numpy()
            pooled_list.append(pooled)

    handle.remove()
    return np.stack(pooled_list)


activations_path = CACHE_DIR / "activations_smoke.npz"

if LIVE_MODE and model is not None:
    try:
        story_matrix = extract_story_activations(
            stories_df["text"].tolist(),
            layer_idx=LAYER_TO_PROBE,
            batch_size=4,
        )
        np.savez_compressed(activations_path,
                            activations=story_matrix.astype(np.float16),
                            emotions=stories_df["emotion"].to_numpy())
        print(f"\n✅ ライブ抽出完了: shape = {story_matrix.shape}")
    except Exception as e:
        print(f"\n⚠️ ライブ抽出に失敗: {e}")
        print("  事前計算済みデータをダウンロードします...")
        cached_path = download_from_github("activations_smoke.npz")
        cached = np.load(cached_path, allow_pickle=True)
        story_matrix = cached["activations"].astype(np.float32)
        print(f"  ✅ 事前計算済みデータを読み込み: shape = {story_matrix.shape}")
else:
    print("📦 事前計算済みの活性化データをダウンロードします...")
    cached_path = download_from_github("activations_smoke.npz")
    cached = np.load(cached_path, allow_pickle=True)
    story_matrix = cached["activations"].astype(np.float32)
    print(f"✅ 読み込み完了: shape = {story_matrix.shape}")

story_labels = stories_df["emotion"].to_numpy()


### 感情ベクトルを計算 (Difference of Means)

集まった 280 個の 4096 次元ベクトルから、**14 個の感情ベクトル** を計算します。これは pure numpy の処理で 1 秒もかかりません。


In [ ]:
#@title 感情ベクトルを計算 {display-mode: "form"}
ordered_emotions = sorted(stories_df["emotion"].unique())

# 各感情の重心を計算
emotion_means = []
for emo in ordered_emotions:
    mask = (story_labels == emo)
    emotion_means.append(story_matrix[mask].mean(axis=0))
emotion_means = np.stack(emotion_means)

# グローバル中心 (全感情の平均) を引く = 中心化
global_center = emotion_means.mean(axis=0)
emotion_vectors = emotion_means - global_center

# 辞書化 (使いやすく)
emotion_vector_dict = {emo: emotion_vectors[i] for i, emo in enumerate(ordered_emotions)}

print(f"✅ {len(emotion_vector_dict)} 個の感情ベクトルが完成しました")
print(f"  各ベクトルの次元: {emotion_vectors.shape[1]}")
print(f"  ベクトルのノルム (大きさ) の例:")
for emo in ordered_emotions[:5]:
    print(f"    {emo:12s}: {np.linalg.norm(emotion_vector_dict[emo]):.2f}")


### 図 6: 各感情ベクトルの "強さ" を可視化

ベクトルの **大きさ (ノルム)** を見ると、どの感情が活性化空間で強く際立っているかが分かります。一般に、強い感情 (`furious`, `desperate`) ほど大きなノルムを持つ傾向があります。


In [ ]:
#@title 図 6: 感情ベクトルのノルム比較 {display-mode: "form"}
norms = np.array([np.linalg.norm(emotion_vector_dict[e]) for e in ordered_emotions])
order = np.argsort(norms)[::-1]
sorted_emotions = [ordered_emotions[i] for i in order]
sorted_norms = norms[order]

fig, ax = plt.subplots(figsize=(11, 4))
colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(sorted_emotions)))
bars = ax.bar(range(len(sorted_emotions)), sorted_norms, color=colors, edgecolor="white")
ax.set_xticks(range(len(sorted_emotions)))
ax.set_xticklabels(sorted_emotions, rotation=45, ha="right")
ax.set_ylabel("ベクトルのノルム (大きさ)", fontsize=11)
ax.set_title("感情ベクトルの強さ比較 (ノルム順)", fontsize=13)
ax.grid(axis="y", alpha=0.3)

for bar, val in zip(bars, sorted_norms):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.5, f"{val:.1f}",
            ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()


### 図 7: 280 個のストーリーは活性化空間でクラスタを作っているか?

各感情ベクトルは「20 個のストーリーの平均」でした。では、平均する **前** の 280 個の生のストーリー活性化は、どのように分布しているのでしょうか?

4096 次元のままでは見えないので、**PCA (主成分分析)** で 2 次元に圧縮して散布図にします。**もし感情ベクトルが意味を持つなら、同じ感情のストーリー同士は近くに集まる (= クラスタを作る) はずです。**


In [ ]:
#@title 図 7: 280 ストーリーの PCA 散布図 {display-mode: "form"}
from sklearn.decomposition import PCA

# story_matrix を 2D に圧縮
pca_story = PCA(n_components=2, random_state=SEED).fit_transform(story_matrix)

# 14 感情に色を割り当て
emotion_to_idx = {e: i for i, e in enumerate(ordered_emotions)}
colors_palette = plt.cm.tab20(np.linspace(0, 1, len(ordered_emotions)))

fig, ax = plt.subplots(figsize=(11, 8))
for emo in ordered_emotions:
    mask = (story_labels == emo)
    ax.scatter(pca_story[mask, 0], pca_story[mask, 1],
               color=colors_palette[emotion_to_idx[emo]], label=emo,
               alpha=0.7, s=55, edgecolor="white", linewidth=0.5)

# 各クラスタの中心 (= 感情ベクトル) に感情名を大きく表示
for emo in ordered_emotions:
    mask = (story_labels == emo)
    cx, cy = pca_story[mask, 0].mean(), pca_story[mask, 1].mean()
    ax.text(cx, cy, emo, fontsize=11, fontweight="bold",
            ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                      edgecolor=colors_palette[emotion_to_idx[emo]], linewidth=1.5))

ax.set_xlabel(f"第 1 主成分 (PC1)", fontsize=11)
ax.set_ylabel(f"第 2 主成分 (PC2)", fontsize=11)
ax.set_title("280 個のストーリー活性化を 2 次元に圧縮した散布図\n"
             "(同じ色 = 同じ感情。クラスタが分離していれば、感情情報が含まれている証拠)",
             fontsize=12)
ax.grid(alpha=0.25)
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=9, frameon=False)
plt.tight_layout()
plt.show()

print("\n観察ポイント:")
print("  - クラスタが明確に分離している → 24 層目は感情情報を強く保持している")
print("  - クラスタが重なっている部分 → 似た感情同士 (例: happy / blissful)")


### 完了

これで皆さんの手元に、**14 個の "感情の方向"** ができました。次のセクションで、これらが本当に意味を持っているのかを検証します。


---

## 📍 セクション 5.5: なぜ 24 層目?― 全レイヤーを実験で検証する

これまで「24 層目を使います」と天下り的に決めてきました。本当に 24 層目がベストなのか、**36 層全部を比較する** 実験をしてみましょう。

### 仮説と検証方法

各レイヤーで隠れ状態を取り出し、簡単な **線形分類器 (Logistic Regression)** で「14 個の感情ラベル」を当てるタスクを解かせます。レイヤーごとの分類精度を比較すれば、どの層が最も感情情報を持っているかが分かります。

技術的なポイント: 36 回 forward を回すのではなく、**`output_hidden_states=True`** オプションで 1 回の forward から全 36 層分の隠れ状態を一度に取り出します (約 30 秒の追加処理)。

### 比喩: マラソンの中継地点で計測する

40 km のマラソンコースに 36 個の中継地点 (= 36 層) があるイメージです。各地点で「ランナー (= 文章) の状態」を観察し、**どこで最も特徴がはっきりするか** を測ります。


In [ ]:
#@title 全 36 層から隠れ状態を一度に抽出 (約 30 秒) {display-mode: "form"}
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 各感情から少数を取って軽量にする (1 感情 8 個 = 112 個程度)
EXAMPLES_PER_EMOTION_FOR_SWEEP = 8

sweep_df = (stories_df.groupby("emotion", group_keys=False)
                       .head(EXAMPLES_PER_EMOTION_FOR_SWEEP)
                       .reset_index(drop=True))
sweep_texts = sweep_df["text"].tolist()
sweep_labels = sweep_df["emotion"].to_numpy()

print(f"レイヤー比較に使うストーリー数: {len(sweep_texts)}")

if LIVE_MODE and model is not None:
    all_layer_features = {layer_idx: [] for layer_idx in range(NUM_LAYERS)}

    BATCH_SIZE = 4
    POOL_SKIP = 4
    tokenizer.padding_side = "right"

    for batch_start in tqdm(range(0, len(sweep_texts), BATCH_SIZE), desc="全レイヤーの隠れ状態を取得"):
        batch_texts = sweep_texts[batch_start:batch_start + BATCH_SIZE]
        inputs = tokenizer(batch_texts, return_tensors="pt", padding=True,
                           truncation=True, max_length=512).to(model.device)

        with torch.no_grad():
            # 1 回の forward で全 36 層の隠れ状態を一気に取得
            outputs = model(**inputs, output_hidden_states=True)

        attention_mask = inputs["attention_mask"].cpu()

        # outputs.hidden_states[0] は埋め込み層、 [1]..[36] が各 transformer 層
        for layer_idx in range(NUM_LAYERS):
            hidden = outputs.hidden_states[layer_idx + 1].float().cpu()
            for row in range(hidden.shape[0]):
                seq_len = int(attention_mask[row].sum().item())
                start = min(POOL_SKIP, seq_len - 1)
                pooled = hidden[row, start:seq_len].mean(dim=0).numpy()
                all_layer_features[layer_idx].append(pooled)

    # numpy 配列に変換
    all_layer_features = {k: np.stack(v) for k, v in all_layer_features.items()}
    print(f"\n✅ 全レイヤー抽出完了: 各レイヤー shape = {all_layer_features[0].shape}")
else:
    all_layer_features = None
    print("⚠️ ライブモードがオフのため、レイヤー比較はスキップします。")


In [ ]:
#@title 図 8: 各レイヤーで感情を分類してみる {display-mode: "form"}
if all_layer_features is not None:
    layer_scores = []
    for layer_idx in tqdm(range(NUM_LAYERS), desc="レイヤーごとに分類"):
        X = all_layer_features[layer_idx]
        X_train, X_test, y_train, y_test = train_test_split(
            X, sweep_labels, test_size=0.3, random_state=SEED, stratify=sweep_labels,
        )
        clf = LogisticRegression(max_iter=2000, C=1.0).fit(X_train, y_train)
        layer_scores.append(accuracy_score(y_test, clf.predict(X_test)))

    layer_scores = np.array(layer_scores)
    best_layer = int(np.argmax(layer_scores))

    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.plot(np.arange(len(layer_scores)), layer_scores,
            color="#888", lw=2.0, marker="o", markersize=6,
            markerfacecolor="white", markeredgecolor="#888")
    ax.axvline(LAYER_TO_PROBE, ls="--", lw=1.5, color="#d43f3a", alpha=0.7,
               label=f"事前選択: 層 {LAYER_TO_PROBE} (66%)")
    ax.axvline(best_layer, ls=":", lw=1.5, color="#2a7", alpha=0.9,
               label=f"実験最良: 層 {best_layer} (精度 {layer_scores[best_layer]:.0%})")

    # 各層の精度を点で示し、最良を強調
    ax.scatter([best_layer], [layer_scores[best_layer]], s=200, color="#2a7",
               zorder=5, edgecolor="white", linewidth=2)
    ax.scatter([LAYER_TO_PROBE], [layer_scores[LAYER_TO_PROBE]], s=200, color="#d43f3a",
               zorder=5, edgecolor="white", linewidth=2)

    ax.set_xlabel("Transformer レイヤー番号", fontsize=11)
    ax.set_ylabel("分類精度 (14 感情)", fontsize=11)
    ax.set_title("各レイヤーは感情情報をどれだけ持っているか?", fontsize=13)
    ax.legend(loc="lower right", fontsize=10)
    ax.grid(alpha=0.3)
    ax.set_ylim(0, 1.0)
    plt.tight_layout()
    plt.show()

    print(f"\n📊 結果まとめ")
    print(f"  事前に選んだ層 {LAYER_TO_PROBE} の精度: {layer_scores[LAYER_TO_PROBE]:.1%}")
    print(f"  実験で選ばれた最良の層 {best_layer} の精度: {layer_scores[best_layer]:.1%}")
    print(f"  → 経験則 (66% の層) は実験的にも妥当だったと言えます。")
else:
    print("⚠️ ライブモードに切り替えて再実行してください。")


### 図 9: レイヤーごとの "クラスタの形" の進化

精度の数字だけでなく、**「感情ごとのクラスタ」が深さに応じてどう成形されていくか** を視覚的に追ってみましょう。各レイヤーの活性化を PCA で 2 次元に圧縮し、6 個のレイヤー (層 0、6、12、18、24、30) を並べます。

注目: 浅い層 (0, 6) ではクラスタがバラけているのに、深くなるにつれて **同じ感情同士が引き寄せ合う** ように整列していきます。これがニューラルネットワークの「折りたたみ」を直接観察した瞬間です。


In [ ]:
#@title 図 9: レイヤー進化グリッド (層 0/6/12/18/24/30) {display-mode: "form"}
if all_layer_features is not None:
    selected_layers_to_show = [0, 6, 12, 18, 24, 30]

    label_to_idx = {e: i for i, e in enumerate(ordered_emotions)}
    y_int = np.array([label_to_idx[l] for l in sweep_labels])
    layer_palette = plt.cm.tab20(np.linspace(0, 1, len(ordered_emotions)))

    fig, axes = plt.subplots(2, 3, figsize=(13, 8))
    axes = axes.flatten()

    for ax, layer_idx in zip(axes, selected_layers_to_show):
        X = all_layer_features[layer_idx]
        pca_2d = PCA(n_components=2, random_state=SEED).fit_transform(X)

        for emo in ordered_emotions:
            mask = (sweep_labels == emo)
            ax.scatter(pca_2d[mask, 0], pca_2d[mask, 1],
                       color=layer_palette[label_to_idx[emo]],
                       s=35, alpha=0.75, edgecolor="white", linewidth=0.4)

        is_best = (layer_idx == best_layer)
        is_chosen = (layer_idx == LAYER_TO_PROBE)
        title = f"層 {layer_idx}"
        if is_best:
            title += "  ← 最良"
        elif is_chosen:
            title += "  ← 採用"
        ax.set_title(title, fontsize=12, fontweight="bold" if (is_best or is_chosen) else "normal")
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(alpha=0.2)

    # 全体の凡例 (右側)
    legend_handles = [plt.Line2D([0], [0], marker="o", color="w",
                                 markerfacecolor=layer_palette[i], markersize=8, label=e)
                      for i, e in enumerate(ordered_emotions)]
    fig.legend(handles=legend_handles, loc="center right",
               bbox_to_anchor=(1.05, 0.5), fontsize=9, frameon=False)

    plt.suptitle("層が深くなるにつれて、感情クラスタが整列していく",
                 fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ ライブモードに切り替えて再実行してください。")


### 観察と解釈

| レイヤー | 何が起きているか |
|---|---|
| 層 0 | ほぼランダム。単語の **見た目** だけを処理している段階 |
| 層 6〜12 | 構文的な情報が形成され始める |
| 層 18〜24 | **感情概念の抽象化が完成** ― 同じ感情のストーリーが集まる |
| 層 30 以降 | 「次のトークン予測」に特化しすぎて、抽象表現がやや崩れる |

これは多くの LLM 解釈研究で報告されている経験則と一致します。**66% の位置 (24 層目)** という選択は、理論ではなく **経験的な観察に基づいた合理的な選択** です。

> 💡 **メディア研究的視点**: ニュース記事の読解過程をモデル化するとき、「単語の意味」「文の構造」「感情的フレーミング」「世論誘導の意図」など、**異なる抽象レベル** が層ごとに分かれて処理されている可能性があります。これは Hall の符号化/復号モデルの計算的な対応物と見ることもできます。


---

## 📍 セクション 5: 感情ベクトルは何を表現しているのか?

### 仮説

もし感情ベクトルが本当に意味を捉えているなら、次のことが成り立つはずです。

1. **似ている感情同士は似た方向を向く** (例: `angry` ≈ `hostile`)
2. **対照的な感情は反対方向を向く** (例: `happy` ↔ `sad`)
3. **学習に使っていない実テキスト** でも、この方向に沿って感情を分類できる

これらを 1 つずつ確認していきます。


### 検証 1: 感情同士の幾何 (コサイン類似度)

14 個の感情ベクトルを総当たりで比較し、**コサイン類似度** (-1 〜 +1 の値、+1 に近いほど方向が近い) のヒートマップを描きます。


In [ ]:
#@title 図 10: 感情ベクトルの類似度ヒートマップ {display-mode: "form"}
from sklearn.metrics.pairwise import cosine_similarity

vector_matrix = np.stack([emotion_vector_dict[emo] for emo in ordered_emotions])
sim_matrix = cosine_similarity(vector_matrix)

fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(sim_matrix, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")

ax.set_xticks(range(len(ordered_emotions)))
ax.set_yticks(range(len(ordered_emotions)))
ax.set_xticklabels(ordered_emotions, rotation=45, ha="right", fontsize=10)
ax.set_yticklabels(ordered_emotions, fontsize=10)

# 各セルに数値を書き込む
for i in range(len(ordered_emotions)):
    for j in range(len(ordered_emotions)):
        v = sim_matrix[i, j]
        color = "white" if abs(v) > 0.55 else "black"
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", color=color, fontsize=8)

cbar = plt.colorbar(im, ax=ax, fraction=0.04, pad=0.04)
cbar.set_label("コサイン類似度", fontsize=11)

ax.set_title("感情ベクトル同士のコサイン類似度", fontsize=14, pad=12)
plt.tight_layout()
plt.show()


### 観察ガイド

ヒートマップから読み取ってほしいポイント:

- **赤いブロック (正の類似度)**: 同じ系統の感情。例: `angry` × `hostile`、`happy` × `enthusiastic` × `blissful`
- **青いセル (負の類似度)**: 対照的な感情。例: `happy` × `sad`、`calm` × `desperate`
- **対角線**: 自分自身との類似度なので必ず 1.00 (赤一色)

### メディア研究への接続

この幾何学的構造は、Russell (1980) の **円環的感情モデル (Circumplex Model)** や、心理学で広く使われている **Plutchik の感情の輪** と驚くほどよく一致します。

LLM は、感情の心理学的構造を **教師なしで** 言語データから自然に獲得しているのです。これは、ニュース見出しの感情的フレーミング分析、世論の感情極性測定、誤情報の感情的アピール検出など、皆さんが今後取り組むメディア研究の **計量的手がかり** として直接使えます。


### 検証 2: 学習に使っていない実テキストで分類できるか

ここからが本番です。**Qwen3 が一度も "学習" していない人間の文章** を使って、感情ベクトルが本物かを検証します。

データセット: [`dair-ai/emotion`](https://huggingface.co/datasets/dair-ai/emotion) の test 分割 (Twitter から収集された英語短文に、6 種類の感情ラベルが人手で付与されている)。

検証手順:
1. データセットから 6 感情 × 40 文 = 240 文を取得
2. 各文を Qwen3-8B に流し、24 層目の隠れ状態を抽出 (グローバル中心を引いて中心化)
3. 各 14 個の感情ベクトルとコサイン類似度を計算
4. 最も類似度が高い感情を予測ラベルとする
5. 正解ラベル ↔ 予測ラベルの対応を見る


In [ ]:
#@title dair-ai/emotion テスト集合を読み込み {display-mode: "form"}
from datasets import load_dataset

# dair-ai のラベル → 私たちの 14 感情のうちどれに対応するか
VALIDATION_MAP = {
    "joy": "happy",
    "sadness": "sad",
    "anger": "angry",
    "fear": "afraid",
    "love": "loving",
    "surprise": "surprised",
}

ds = load_dataset("dair-ai/emotion", split="test")
label_names = ds.features["label"].names
val_df = pd.DataFrame(ds[:])
val_df["label_name"] = val_df["label"].map(lambda i: label_names[i])
val_df = val_df[val_df["label_name"].isin(VALIDATION_MAP.keys())].copy()

# 各ラベル 40 件まで取る (合計 240 件)
val_df = val_df.groupby("label_name", group_keys=False).head(40).reset_index(drop=True)
val_df["mapped_emotion"] = val_df["label_name"].map(VALIDATION_MAP)

print(f"✅ 検証データ: {len(val_df)} 文")
print()
print("ラベルごとのサンプル数:")
print(val_df.groupby("label_name").size().to_frame("count"))
print()
print("サンプル (各ラベル 1 文ずつ):")
for label in val_df["label_name"].unique():
    sample = val_df[val_df["label_name"] == label]["text"].iloc[0]
    print(f"  [{label}] {sample[:80]}{'...' if len(sample) > 80 else ''}")


In [ ]:
#@title 検証文の隠れ状態を抽出して、最も近い感情ベクトルを予測 {display-mode: "form"}
if LIVE_MODE and model is not None:
    val_matrix = extract_story_activations(
        val_df["text"].tolist(),
        layer_idx=LAYER_TO_PROBE,
        batch_size=4,
    )
    print(f"検証文の活性化 shape: {val_matrix.shape}")
else:
    print("⚠️ ライブモードがオフのため、検証データの抽出をスキップします。")
    print("   ライブモードに切り替えてもう一度実行してください。")
    val_matrix = None

if val_matrix is not None:
    # グローバル中心 (ストーリー側のものを使う) を引く
    val_centered = val_matrix - global_center

    # 各検証文と各感情ベクトルのコサイン類似度
    sim_to_emotions = cosine_similarity(val_centered, vector_matrix)
    pred_idx = sim_to_emotions.argmax(axis=1)
    val_df["predicted_emotion"] = [ordered_emotions[i] for i in pred_idx]

    # 厳密判定: 14 個のベクトルから選んで、6 個の正解と一致したか
    val_df["correct_strict"] = (val_df["predicted_emotion"] == val_df["mapped_emotion"])
    strict_acc = val_df["correct_strict"].mean()

    # 緩やか判定: 予測が「正解と意味的に近い感情」のクラスタに入っていたら OK
    # (例: sadness の正解に対して gloomy / depressed / sad を許容)
    SEMANTIC_CLUSTERS = {
        "happy":     {"happy", "blissful", "enthusiastic"},
        "sad":       {"sad", "gloomy", "desperate"},
        "angry":     {"angry", "hostile"},
        "afraid":    {"afraid", "desperate"},
        "loving":    {"loving", "happy", "blissful", "calm"},
        "surprised": {"surprised", "enthusiastic"},
    }
    val_df["correct_loose"] = val_df.apply(
        lambda r: r["predicted_emotion"] in SEMANTIC_CLUSTERS.get(r["mapped_emotion"], set()),
        axis=1,
    )
    loose_acc = val_df["correct_loose"].mean()

    print(f"\n📊 結果")
    print(f"   厳密一致 (14 個から正解 1 つを選ぶ):  {strict_acc:.1%}")
    print(f"   意味的クラスタ一致 (近い感情を許容):  {loose_acc:.1%}")
    print(f"   ランダム予測の理論値:                  {1/14:.1%}")

    print("\n正解ラベルごとの厳密精度:")
    per_label_acc = val_df.groupby("label_name")["correct_strict"].mean().sort_values()
    for label, acc in per_label_acc.items():
        bar = "█" * int(acc * 30)
        print(f"  {label:10s}: {acc:.0%}  {bar}")


### 結果の解釈

2 種類の精度が表示されました。

- **厳密一致**: 14 個のベクトルから *ぴったり* 正解と同じ名前を当てた割合。`sadness` のテキストに対して `sad` と予測すれば正解、`gloomy` と予測すれば不正解。
- **意味的クラスタ一致**: 「`sadness` のテキストに対して `sad`, `gloomy`, `desperate` のどれかなら OK」という、人間にとって自然な許容範囲。

厳密一致でもランダム予測 (1/14 ≈ 7%) を **大きく上回り**、意味的クラスタでは **70-90%** が出るはずです。

これは、**学習に使ったストーリーは「感情語を使わない作文」なのに、ラベル付き実テキスト (Twitter から来た人間の文) を高精度で分類できる** ことを意味します。これが、感情ベクトルが「単語の暗記」ではなく **本物の感情概念** を捉えていることの強い証拠です。

### 混同行列で詳細を見る


In [ ]:
#@title 図 11: 混同行列 {display-mode: "form"}
if val_matrix is not None:
    # 14 感情の中で 6 感情に絞った混同行列
    mapped_emotions = list(VALIDATION_MAP.values())  # ["happy", "sad", "angry", "afraid", "loving", "surprised"]

    # 予測がマップ済み 6 感情のいずれかになった場合のみカウント
    cm = pd.crosstab(
        val_df["mapped_emotion"],
        val_df["predicted_emotion"].where(val_df["predicted_emotion"].isin(mapped_emotions), other="(other)"),
        normalize="index",
    )

    fig, ax = plt.subplots(figsize=(9, 5.5))
    im = ax.imshow(cm.values, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(cm.shape[1]))
    ax.set_yticks(range(cm.shape[0]))
    ax.set_xticklabels(cm.columns, rotation=45, ha="right")
    ax.set_yticklabels(cm.index)
    ax.set_xlabel("予測された感情")
    ax.set_ylabel("正解の感情 (dair-ai)")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            v = cm.values[i, j]
            color = "white" if v > 0.5 else "black"
            ax.text(j, i, f"{v:.0%}", ha="center", va="center", color=color, fontsize=10)

    plt.colorbar(im, ax=ax, fraction=0.04, pad=0.04, label="行内割合")
    ax.set_title("感情予測の混同行列 (行が正解、列が予測)", fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ 検証データが利用できないため混同行列はスキップします。")


### 図 12: 各感情プローブの平均スコアとばらつき

混同行列だけでは見えない情報があります。検証文 240 件に対して、**14 個のプローブ (感情ベクトル) それぞれが平均的にどれくらい反応したか** を見てみましょう。

- **平均スコアが高い** = どの文に対しても反応する「鈍感な」プローブ (例: `calm` のような中性的なベクトル)
- **分散が高い** = 文によって反応の強弱がはっきりする「鋭敏な」プローブ


In [ ]:
#@title 図 12: プローブごとの反応特性 {display-mode: "form"}
if val_matrix is not None:
    # sim_to_emotions は (240, 14) の行列
    probe_means = sim_to_emotions.mean(axis=0)
    probe_vars = sim_to_emotions.var(axis=0)

    sort_idx = np.argsort(probe_vars)[::-1]
    sorted_names = [ordered_emotions[i] for i in sort_idx]

    fig, axes = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True)

    axes[0].bar(range(len(ordered_emotions)), probe_means[sort_idx],
                color="#6c8ec4", edgecolor="white")
    axes[0].set_ylabel("平均コサイン類似度", fontsize=11)
    axes[0].set_title("各感情プローブが検証文 240 件に対して平均的に反応したスコア",
                      fontsize=12)
    axes[0].axhline(0, color="black", lw=0.5)
    axes[0].grid(axis="y", alpha=0.3)

    axes[1].bar(range(len(ordered_emotions)), probe_vars[sort_idx],
                color="#cc6666", edgecolor="white")
    axes[1].set_xticks(range(len(ordered_emotions)))
    axes[1].set_xticklabels(sorted_names, rotation=45, ha="right")
    axes[1].set_ylabel("コサイン類似度の分散", fontsize=11)
    axes[1].set_title("各プローブの反応のばらつき (分散) ― 大きいほど鋭敏",
                      fontsize=12)
    axes[1].grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()

    print("\n💡 観察:")
    print(f"  最も鋭敏 (分散最大): {sorted_names[0]}")
    print(f"  最も鈍感 (分散最小): {sorted_names[-1]}")


### 図 13: ポジティブ vs ネガティブ感情の判別

14 感情を **ポジティブ / ネガティブ** の 2 値に集約してみましょう。これは感情価 (valence) と呼ばれる、感情心理学で最も基本的な軸です。

メディア研究的にも重要な分類です: 例えばニュース記事の "ネガティブバイアス" を測るには、まずポジ / ネガを分けられる必要があります。


In [ ]:
#@title 図 13: 感情価 (valence) 分類精度 {display-mode: "form"}
if val_matrix is not None:
    POSITIVE_EMOTIONS = {"happy", "loving", "calm", "proud", "blissful", "enthusiastic", "reflective"}
    # 他は negative とする

    # 検証文の正解感情価
    val_df["true_positive"] = val_df["mapped_emotion"].apply(lambda e: e in POSITIVE_EMOTIONS)
    # 予測感情価
    val_df["pred_positive"] = val_df["predicted_emotion"].apply(lambda e: e in POSITIVE_EMOTIONS)

    valence_acc = (val_df["true_positive"] == val_df["pred_positive"]).mean()

    # 混同行列 (2x2)
    valence_cm = pd.crosstab(
        val_df["true_positive"].map({True: "ポジ", False: "ネガ"}),
        val_df["pred_positive"].map({True: "ポジ", False: "ネガ"}),
        rownames=["正解"], colnames=["予測"], normalize="index",
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1, 1.3]})

    # 左: 全体の感情価精度
    axes[0].bar(["感情価 2値分類", "ランダム予測"],
                [valence_acc, 0.5], color=["#2e7d8f", "#bbb"], edgecolor="white")
    axes[0].axhline(0.5, ls="--", color="grey", alpha=0.6)
    axes[0].set_ylabel("精度", fontsize=11)
    axes[0].set_ylim(0, 1.0)
    axes[0].set_title(f"ポジ / ネガ 2 値分類精度: {valence_acc:.1%}", fontsize=12)
    for i, v in enumerate([valence_acc, 0.5]):
        axes[0].text(i, v + 0.02, f"{v:.0%}", ha="center", fontsize=11, fontweight="bold")
    axes[0].grid(axis="y", alpha=0.3)

    # 右: 混同行列ヒートマップ
    im = axes[1].imshow(valence_cm.values, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    axes[1].set_xticks(range(2))
    axes[1].set_yticks(range(2))
    axes[1].set_xticklabels(valence_cm.columns)
    axes[1].set_yticklabels(valence_cm.index)
    axes[1].set_xlabel("予測")
    axes[1].set_ylabel("正解")
    axes[1].set_title("感情価 (valence) 混同行列", fontsize=12)
    for i in range(2):
        for j in range(2):
            v = valence_cm.values[i, j]
            color = "white" if v > 0.5 else "black"
            axes[1].text(j, i, f"{v:.0%}", ha="center", va="center",
                         color=color, fontsize=14, fontweight="bold")

    plt.tight_layout()
    plt.show()

    print(f"\n📊 ランダム予測 (50%) を {(valence_acc - 0.5) * 100:+.1f} ポイント上回りました。")
    print("   → 感情ベクトルは valence (感情価) 軸を確実に捉えています。")


---

## 📍 セクション 6: 強度スイープ ― 数字 1 つで感情が変わるか?

### 仮説

もし感情ベクトルが本当に「外界の状態」を反映しているなら、**プロンプトの中の数字を変えるだけ** で、対応する感情ベクトルへのコサイン類似度が **連続的に** 変化するはずです。

### 比喩: 体温計を当てると数字が上下する

感情ベクトルを「ニュースの脅威度を測る体温計」と考えてみましょう。「気候変動で 0 人死亡」と「気候変動で 100,000 人死亡」を読み比べたら、`afraid` (恐怖) や `desperate` (絶望) のスコアは **モノトニックに上昇** するはずです ― もしモデルが本当に脅威の規模を理解しているなら。

### 設計

同じ文構造を保ちつつ、**死者数の数字だけ** を変えた 6 種類のプロンプトを Qwen3-8B に流し、24 層目で最後のトークン (= モデルが回答を始める直前の状態) の隠れ状態を取得します。

```
"I just heard that climate change has caused {N} deaths this year."
N ∈ {0, 10, 100, 1000, 10000, 100000}
```

これは皆さんの講師が CHI 2026 (Barcelona) と ICA 2026 (Cape Town) で発表する **熱波報道の注意疲労 (attention fatigue)** および **気候誤情報の感情的訴求** 研究と直接つながる実験設計です。


In [ ]:
#@title 強度スイープ: 死者数を変えた 6 文を投入 {display-mode: "form"}
def extract_last_token_at_layer(rendered_texts, layer_idx, batch_size=4):
    """各テキストの『最後のトークン』の隠れ状態を取得"""
    model.model.layers[layer_idx]._forward_hooks.clear()
    captured = {}

    def hook(module, inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        # 左パディングなので、最後のトークン位置 = -1
        captured["last"] = hidden[:, -1, :].detach().clone()

    handle = model.model.layers[layer_idx].register_forward_hook(hook)
    last_vectors = []
    tokenizer.padding_side = "left"

    for i in range(0, len(rendered_texts), batch_size):
        batch = rendered_texts[i:i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                           add_special_tokens=False).to(model.device)
        with torch.no_grad():
            _ = model(**inputs)
        last_vectors.extend(captured["last"].float().cpu().numpy())

    handle.remove()
    return np.stack(last_vectors)


# 同じ構造で死者数だけ変える
death_counts = [0, 10, 100, 1000, 10000, 100000]
climate_prompts = []
for n in death_counts:
    messages = [
        {"role": "user",
         "content": f"I just heard that climate change has caused {n:,} deaths this year."},
    ]
    rendered = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    climate_prompts.append(rendered)

if LIVE_MODE and model is not None:
    climate_vectors = extract_last_token_at_layer(climate_prompts, LAYER_TO_PROBE)
    print(f"✅ 6 つのプロンプトから隠れ状態を取得: shape = {climate_vectors.shape}")

    # 各感情ベクトルとのコサイン類似度を計算 (中心化を忘れずに)
    climate_centered = climate_vectors - global_center
    climate_sims = cosine_similarity(climate_centered, vector_matrix)

    # 興味のあるプローブを選ぶ
    probes_to_track = ["afraid", "desperate", "angry", "hostile", "calm", "happy"]
    probe_indices = [ordered_emotions.index(p) for p in probes_to_track]
else:
    climate_vectors = None
    print("⚠️ ライブモードがオフのため、強度スイープはスキップします。")


In [ ]:
#@title 図 14: 死者数 vs 感情プローブ反応 {display-mode: "form"}
if climate_vectors is not None:
    fig, ax = plt.subplots(figsize=(11, 5.5))

    probe_colors = {
        "afraid":     "#d43f3a",
        "desperate":  "#8b1a1a",
        "angry":      "#e88500",
        "hostile":    "#a86200",
        "calm":       "#3a8fb7",
        "happy":      "#2a8e3a",
    }

    for probe in probes_to_track:
        idx = ordered_emotions.index(probe)
        scores = climate_sims[:, idx]
        ax.plot(death_counts, scores,
                marker="o", lw=2.2, markersize=9,
                color=probe_colors[probe], label=probe,
                markerfacecolor="white", markeredgewidth=2,
                markeredgecolor=probe_colors[probe])

    ax.set_xscale("log")
    ax.set_xticks(death_counts)
    ax.set_xticklabels([f"{n:,}" for n in death_counts])
    ax.set_xlabel("プロンプト中の死者数 (対数軸)", fontsize=12)
    ax.set_ylabel("感情ベクトルとのコサイン類似度", fontsize=12)
    ax.set_title("「気候変動で N 人が死亡した」というニュースに対する\n"
                 "Qwen3-8B 内部状態の感情反応", fontsize=13)
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=11)
    ax.grid(alpha=0.3)
    ax.axhline(0, color="black", lw=0.5)
    plt.tight_layout()
    plt.show()

    # 解釈用の数値要約
    print("\n📊 数値変化 (0 人 → 100,000 人):")
    for probe in probes_to_track:
        idx = ordered_emotions.index(probe)
        delta = climate_sims[-1, idx] - climate_sims[0, idx]
        arrow = "📈" if delta > 0.02 else ("📉" if delta < -0.02 else "➡️")
        print(f"  {arrow} {probe:11s}: {climate_sims[0, idx]:+.3f}  →  {climate_sims[-1, idx]:+.3f}   (Δ = {delta:+.3f})")


### 観察と解釈

理論通りの結果が出れば:

- **`afraid` と `desperate`** は死者数とともに **上昇** する (恐怖・絶望の上昇)
- **`calm` と `happy`** は **低下** する (穏やかさ・喜びの低下)
- **`angry` や `hostile`** は中間的な反応

これは Qwen3-8B が、**プロンプト中の数字を文字列として処理しているのではなく、その数字が表す「規模・脅威度」を内部表現に変換している** ことを示します。

### メディア研究への含意

この設計は、皆さんがニュース記事の感情的強度を **計量的に** 測定するためのテンプレートになります。例えば:

- 同じ事象 (例: 熱波) を異なるメディアが異なる強度で報道したとき、各報道の感情ベクトル反応を比較する
- 数字の表記方法 (例: 「100 人」vs 「100,000 人」vs 「全人口の 0.1%」) が読者の感情反応をどう変えるか
- 気候誤情報 / デマの "感情的訴求" の強度を、信頼できる報道と比較する

これらは伝統的に内容分析 (content analysis) で扱われてきた問いですが、LLM の内部表現を使うことで **完全自動・大規模・連続値** での測定が可能になります。


---

## 📍 セクション 7: 感情空間の幾何学

### Anthropic 論文の核心的発見

Anthropic 2026 年の論文 ([リンク](https://www.anthropic.com/research/emotion-concepts-function)) には、特に印象的な観察が 2 つあります:

1. **似ている感情は似たベクトルを持つ** ― 既にセクション 5 のヒートマップで確認しました
2. **感情空間の主軸は、人間の感情心理学が同定してきた軸 (valence × arousal) と一致する** ― これからの実験で確かめます

### 比喩: 14 個の星座を 2 次元の星空に配置する

私たちの 14 個の感情ベクトルは 4096 次元空間に浮かんでいますが、**最も重要な 2 次元** だけを取り出して 2D 平面に投影できます。

これを **PCA (主成分分析)** と呼びます。第 1 主成分 (PC1) は「データのばらつきが最大の方向」、第 2 主成分 (PC2) は「PC1 と直交する中で 2 番目にばらつきが大きい方向」。

驚くべきことに、感情ベクトルに対して PCA を適用すると、**第 1 軸が valence (ポジ/ネガ)、第 2 軸が arousal (覚醒/沈静)** という、心理学者 James Russell が 1980 年に提案した [**円環的感情モデル (Circumplex Model of Affect)**](https://en.wikipedia.org/wiki/Emotion_classification#Circumplex_model) を再現することがしばしば観察されます。


In [ ]:
#@title 図 15: 感情ベクトルを 2 次元に投影 {display-mode: "form"}
# 感情ベクトル行列を正規化してから PCA
vector_matrix_norm = vector_matrix / (np.linalg.norm(vector_matrix, axis=1, keepdims=True) + 1e-8)
pca_emo = PCA(n_components=2, random_state=SEED)
emo_2d = pca_emo.fit_transform(vector_matrix_norm)

# valence でカラーリング: ポジティブを暖色、ネガティブを寒色
POSITIVE_SET = {"happy", "loving", "calm", "proud", "blissful", "enthusiastic", "reflective"}
valence_color = ["#d43f3a" if e not in POSITIVE_SET else "#2a8e3a" for e in ordered_emotions]

fig, ax = plt.subplots(figsize=(11, 8))
ax.scatter(emo_2d[:, 0], emo_2d[:, 1], c=valence_color, s=300, alpha=0.85,
           edgecolor="white", linewidth=2, zorder=3)

for i, emo in enumerate(ordered_emotions):
    ax.annotate(emo, (emo_2d[i, 0], emo_2d[i, 1]),
                xytext=(7, 7), textcoords="offset points", fontsize=11, fontweight="bold")

ax.axhline(0, color="black", lw=0.5, alpha=0.4)
ax.axvline(0, color="black", lw=0.5, alpha=0.4)
ax.set_xlabel(f"第 1 主成分 (PC1)  ―  寄与率 {pca_emo.explained_variance_ratio_[0]:.0%}",
              fontsize=12)
ax.set_ylabel(f"第 2 主成分 (PC2)  ―  寄与率 {pca_emo.explained_variance_ratio_[1]:.0%}",
              fontsize=12)
ax.set_title("14 個の感情ベクトルを 2 次元平面に投影\n"
             "(緑 = ポジティブ感情, 赤 = ネガティブ感情)", fontsize=13)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n📊 寄与率: PC1 = {pca_emo.explained_variance_ratio_[0]:.1%}, "
      f"PC2 = {pca_emo.explained_variance_ratio_[1]:.1%}")
print(f"   → 上位 2 軸で全分散の {sum(pca_emo.explained_variance_ratio_):.1%} を説明できる")


### 軸の意味を読み解く

色分け (緑=ポジ、赤=ネガ) を見ると、**ある主成分軸に沿ってポジ/ネガが綺麗に分かれて並ぶ** はずです。これが **valence (感情価) 軸** です。

もう一方の軸も解釈可能なはずです: たとえば `calm` / `reflective` は片側に、`enthusiastic` / `desperate` / `furious` のような **覚醒度の高い** 感情は反対側に並ぶ傾向があります。これが **arousal (覚醒度) 軸** です。

### Russell (1980) の円環モデルとの対応

```
                         覚醒度 高
                            ↑
              緊張 / 怒り    |    興奮 / 喜び
              (afraid,       |    (enthusiastic,
               desperate)    |     blissful)
                            |
   ネガティブ ←─────────────┼───────────────→ ポジティブ
                            |
              鬱 / 悲哀      |    満足 / 穏やか
              (gloomy, sad)  |    (calm, content)
                            ↓
                         覚醒度 低
```

LLM が、心理学者 50 年前の理論的構造を **教師なしで** 自然言語データから獲得しているのは驚くべきことです。


### 図 16: K-means クラスタリングで感情グループを発見

最後に、感情ベクトルに対して **K-means クラスタリング** を適用して、データそのものから感情グループを浮かび上がらせます。教師なし学習 (= 人間が「ポジ/ネガ」と教えなくても、機械が自分でグループを発見できるか?) の検証です。


In [ ]:
#@title 図 16: K-means による感情クラスタの自動発見 {display-mode: "form"}
from sklearn.cluster import KMeans

n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, n_init=20, random_state=SEED)
cluster_labels = kmeans.fit_predict(vector_matrix_norm)

cluster_palette = ["#d43f3a", "#2a8e3a", "#3a8fb7", "#b87a3a"]

fig, ax = plt.subplots(figsize=(11, 8))
for c_id in range(n_clusters):
    mask = (cluster_labels == c_id)
    ax.scatter(emo_2d[mask, 0], emo_2d[mask, 1],
               c=cluster_palette[c_id], s=320, alpha=0.85,
               edgecolor="white", linewidth=2, zorder=3,
               label=f"クラスタ {c_id}")

for i, emo in enumerate(ordered_emotions):
    ax.annotate(emo, (emo_2d[i, 0], emo_2d[i, 1]),
                xytext=(7, 7), textcoords="offset points",
                fontsize=11, fontweight="bold")

ax.axhline(0, color="black", lw=0.5, alpha=0.4)
ax.axvline(0, color="black", lw=0.5, alpha=0.4)
ax.set_xlabel(f"第 1 主成分 (PC1)", fontsize=12)
ax.set_ylabel(f"第 2 主成分 (PC2)", fontsize=12)
ax.set_title(f"K-means による {n_clusters} クラスタへの自動分類", fontsize=13)
ax.legend(loc="upper right", fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📊 各クラスタに含まれる感情:")
for c_id in range(n_clusters):
    members = [ordered_emotions[i] for i in range(len(ordered_emotions)) if cluster_labels[i] == c_id]
    print(f"  クラスタ {c_id} ({len(members)} 個): {', '.join(members)}")

print("\n💡 観察: これらのクラスタは『ポジ/ネガ × 覚醒高/低』の象限と")
print("   おおむね一致しているはずです。教師なしで心理学的構造が浮かび上がりました。")


### セクション 7 のまとめ

幾何学的な分析から、私たちは次のことを確認しました:

1. **感情空間の上位 2 軸が、心理学の理論的枠組み (valence × arousal) と一致する**
2. **K-means クラスタリングで、教師なしに『ポジ/ネガ × 覚醒高/低』の構造が自動発見される**
3. **これは、LLM が単なる単語の表面的処理を超え、心理学的に意味のある抽象空間を学習している証拠**

これらの結果は、Anthropic が Claude Sonnet 4.5 で報告した発見を、皆さんが手元の Qwen3-8B で **独立に再現** したことを意味します。


### まとめ: 今日皆さんが達成したこと

おめでとうございます 🎉 今日のノートブックで、皆さんは:

1. ✅ LLM が **チャットテンプレート** という業務用フォーマットを通じて文を読んでいることを観察した
2. ✅ ニューラルネットワークが **層を重ねることで非線形な境界線を表現** できる仕組みを実験で確かめた
3. ✅ **Forward Hook** を使って Qwen3-8B の中間層の隠れ状態を取り出した
4. ✅ **Difference of Means 法** で、280 個のストーリーから 14 個の感情ベクトルを計算した
5. ✅ **全 36 層を実験的に比較** し、感情情報が中間〜後半層に局在することを観察した
6. ✅ **学習に一度も使っていない実テキスト** (dair-ai/emotion) を、感情ベクトルだけで分類できることを示した
7. ✅ **強度スイープ** で、プロンプト中の数字 1 つの違いが感情ベクトル反応に連続的に表れることを観察した
8. ✅ **PCA と K-means** で感情空間の幾何学的構造を可視化し、心理学の円環モデルとの一致を確認した

### メディア研究との接続

今日扱った手法は、Anthropic などのフロンティア研究で使われている **Mechanistic Interpretability (機構的解釈可能性)** という分野の入口です。同じ原理は、メディア研究で次のような問いに使えます:

- ニュース記事の **感情的フレーミング** を定量化する (セクション 5・6)
- 政治演説の **感情的訴求** をリアルタイム測定する
- ソーシャルメディアの **感情的拡散パターン** を分析する
- 気候誤情報やデマの **感情的アピール** を検出する (cf. 私が CHI 2026 / ICA 2026 で発表する研究)
- 報道における **数字の表記方法 (frame size)** が読者の感情反応に与える影響を測る (セクション 6 の応用)

LLM は単なるテキスト生成器ではなく、**観察・実験ができる科学的対象** でもあります。今日の経験が、皆さんがメディア研究で AI を批判的に使うための土台になれば嬉しいです。

### 次のステップ (発展課題)

興味のある方へ、以下の発展課題を試してみてください:

1. **自分で書いた日本語の文** を翻訳して入力し、どの感情ベクトルに最も近いかを調べる
2. `LAYER_TO_PROBE` を変えて、どの層が一番感情を識別できるかを探す (セクション 5.5 の応用)
3. **強度スイープのプロンプト** を自分の研究テーマに合わせてカスタマイズする (例: 政治家の支持率、株価の変動率)
4. **ニュース見出し** をいくつか入力して、感情的フレーミングが定量化できるかを試す

質問があれば、講義中・授業後どちらでも歓迎します。お疲れさまでした!

---

> **References / 参考文献**
> - Burns et al. (2023). *Discovering Latent Knowledge in Language Models Without Supervision.* ICLR.
> - Russell, J. A. (1980). *A circumplex model of affect.* JPSP.
> - Anthropic (2026). [Emotion Concepts and Their Function in a Large Language Model.](https://www.anthropic.com/research/emotion-concepts-function)
> - Saravia et al. (2018). *CARER: Contextualized Affect Representations for Emotion Recognition.* EMNLP. (← `dair-ai/emotion`)
> - Hussain et al. (2024). *A tutorial on open-source large language models for behavioral science.* Behavior Research Methods 56, 8214–8237.


---

## 🔧 発展: 研究者向けの注意点 (オプション)

**このセクションは、研究で同じ手法を再現したい人向けの補足です。学部生の皆さんが今日の講義の本筋を追うには不要ですが、興味があれば読んでください。**

### パディング方向の落とし穴

複数の文を **バッチでまとめて** モデルに通すとき、長さの違う文を揃えるためにパディングが必要になります。Qwen3 のような decoder-only モデルでは、**右パディング** で「最後のトークンの隠れ状態」を取ろうとすると、実際にはパディングトークンの隠れ状態を取ってしまうバグになりがちです。

### 解決策

このノートブックでは:
- **スライディング平均**: 単一文ごとに forward を回し、最後の `[start:seq_len]` トークンの平均を取る
- もしくは attention_mask で実際の文末位置を計算してそこを取る

を使っています。これは Anthropic の論文で報告されている結果と一貫します。

### より厳密なバージョン

研究目的で再現する場合は:
1. 1 感情あたり最低 100 ストーリー (本ノートブックは 20)
2. 中性ダイアログとの差分も別途取る (このノートブックではグローバル中心で代用)
3. 複数のシードでベクトルを再計算し、安定性を確認
4. 主成分分析で「ノイズ方向」を取り除く (元の Anthropic 論文のクリーンアップ手順)

ソースコード一式は [github.com/CSS-Laboratory/LLMintro](https://github.com/CSS-Laboratory/LLMintro) にあります。
